# Marketing Campaign Analysis

This notebook walks through data cleaning, feature engineering, exploratory analysis, segmentation, and SQL-ready exports for the marketing campaign dataset.

## 1. Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

data_dir = Path("..")
marketing_path = data_dir / "marketing_data.csv"
dictionary_path = data_dir / "marketing_data_dictionary.csv"

df = pd.read_csv(marketing_path)
dictionary = pd.read_csv(dictionary_path)

print("Loaded marketing_data.csv with shape:", df.shape)
print("Loaded data dictionary with shape:", dictionary.shape)

## 2. Inspect Data Dictionary and Initial Dataset

In [ ]:
display(dictionary.head(15))

display(df.head())
display(df.info())
display(df.describe(include="all"))
print("Missing values by column:")
print(df.isna().sum().sort_values(ascending=False).head(20))

## 3. Clean Data, Handle Missing Values, and Convert Types

In [ ]:
clean = df.copy()

# Convert date fields
clean['Dt_Customer'] = pd.to_datetime(clean['Dt_Customer'], errors='coerce')

# Income cleaning
clean['Income'] = clean['Income'].replace(0, np.nan)
income_median = clean['Income'].median()
clean['Income'] = clean['Income'].fillna(income_median)

# Standardize categorical fields
clean['Education'] = clean['Education'].str.strip().replace({
    'Basic': 'Basic',
    '2n Cycle': '2n Cycle',
    'Graduation': 'Graduation',
    'Master': 'Master',
    'PhD': 'PhD'
})
clean['Marital_Status'] = clean['Marital_Status'].str.strip().replace({
    'Married': 'Married',
    'Together': 'Together',
    'Single': 'Single',
    'Divorced': 'Divorced',
    'Widow': 'Widow',
    'Alone': 'Alone',
    'Absurd': 'Absurd',
    'YOLO': 'YOLO'
})

# Handle invalid ages and birth years
clean['Age'] = 2015 - clean['Year_Birth']
clean.loc[clean['Age'] < 18, 'Age'] = np.nan
clean.loc[clean['Age'] > 100, 'Age'] = np.nan
clean['Age'] = clean['Age'].fillna(clean['Age'].median()).astype(int)

# Preserve customer IDs as integer keys
clean['ID'] = clean['ID'].astype(int)

print('Data types after cleaning:')
print(clean[['Dt_Customer', 'Income', 'Age', 'Education', 'Marital_Status']].dtypes)

## 4. Derive Customer Metrics

In [ ]:
clean['Customer_Tenure_Days'] = (pd.to_datetime('2015-01-01') - clean['Dt_Customer']).dt.days
clean['Customer_Tenure_Months'] = (clean['Customer_Tenure_Days'] / 30).round(1)

spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
clean['Total_Spend'] = clean[spend_cols].sum(axis=1)

purchase_cols = ['NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
clean['Total_Purchases'] = clean[purchase_cols].sum(axis=1)

clean['Children'] = clean['Kidhome'] + clean['Teenhome']
clean['Income_Band'] = pd.cut(
    clean['Income'],
    bins=[0, 30000, 60000, 75000, 100000, np.inf],
    labels=['Low', 'Medium', 'Upper-Mid', 'High', 'Very High'],
    include_lowest=True,
)
clean['Age_Band'] = pd.cut(
    clean['Age'],
    bins=[17, 29, 39, 49, 59, np.inf],
    labels=['18-29', '30-39', '40-49', '50-59', '60+'],
    include_lowest=True,
)

display(clean[['ID', 'Dt_Customer', 'Age', 'Customer_Tenure_Months', 'Total_Spend', 'Total_Purchases', 'Children', 'Income_Band', 'Age_Band']].head())

## 5. Detect and Treat Outliers

In [ ]:
outlier_columns = ['Income', 'Age', 'Total_Spend', 'Customer_Tenure_Months', 'Recency']
for col in outlier_columns:
    q1 = clean[col].quantile(0.01)
    q99 = clean[col].quantile(0.99)
    clean[col] = clean[col].clip(lower=q1, upper=q99)

clean[outlier_columns].describe().T

## 6. Univariate Analysis of Demographics and Spending

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sns.histplot(clean['Age'], kde=True, ax=axes[0, 0], color='tab:blue')
axes[0, 0].set_title('Age Distribution')
sns.histplot(clean['Income'], kde=True, ax=axes[0, 1], color='tab:green')
axes[0, 1].set_title('Income Distribution')
sns.histplot(clean['Total_Spend'], kde=True, ax=axes[0, 2], color='tab:orange')
axes[0, 2].set_title('Total Spend Distribution')
sns.histplot(clean['Customer_Tenure_Months'], kde=True, ax=axes[1, 0], color='tab:purple')
axes[1, 0].set_title('Customer Tenure (Months)')
sns.histplot(clean['Recency'], kde=True, ax=axes[1, 1], color='tab:red')
axes[1, 1].set_title('Recency (Days Since Last Purchase)')
axes[1, 2].axis('off')
plt.tight_layout()
plt.show()

category_spend = clean[spend_cols].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=category_spend.index, y=category_spend.values, palette='Spectral')
plt.title('Average Spend by Product Category')
plt.ylabel('Average Spend')
plt.xlabel('Product Category')
plt.xticks(rotation=45)
plt.show()

**Takeaway:** Most customers are aged 40–60, with income concentrated between 30K–80K. Total spend is right-skewed — a small group spends significantly more. Customer tenure spans 1–3 years with most joining in 2012–2014. Recency is roughly uniform, indicating no strong seasonality in last purchase timing. Wines dominate product-level spending by a wide margin.

## 7. Bivariate and Multivariate Analysis

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x='Response', y='Income', data=clean, palette='Set2')
plt.title('Income by Response')
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(x='Response', y='Age', data=clean, palette='Set3')
plt.title('Age by Response')
plt.show()

spend_vs_response = clean.groupby('Response')[spend_cols].mean().T.reset_index()
spend_vs_response.columns = ['Category', 'No_Response', 'Response']

plt.figure(figsize=(10, 5))
spend_vs_response.plot(x='Category', kind='bar', figsize=(10, 5), color=['#d9534f', '#5cb85c'])
plt.title('Average Spend by Category and Response')
plt.ylabel('Average Spend')
plt.xticks(rotation=45)
plt.legend(title='Response')
plt.tight_layout()
plt.show()

# Marital status vs response rate
marital_response = clean.groupby('Marital_Status')['Response'].mean().sort_values(ascending=False).reset_index()
marital_response['response_rate_pct'] = marital_response['Response'] * 100
plt.figure(figsize=(10, 5))
sns.barplot(x='Marital_Status', y='response_rate_pct', data=marital_response, palette='muted')
plt.title('Campaign Response Rate by Marital Status')
plt.ylabel('Response Rate (%)')
plt.xlabel('Marital Status')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

visit_response = clean.groupby('Response')[['NumWebVisitsMonth', 'NumStorePurchases', 'NumCatalogPurchases', 'NumWebPurchases']].mean().T
display(visit_response)

**Takeaway:** Customers who accepted the final campaign (Response = 1) tend to have higher income and are not systematically older or younger. Responders spend significantly more across all product categories — especially wines and meat — suggesting premium product interest correlates with campaign acceptance. Marital status shows variation in response rates, with some groups (e.g., Married, Together) potentially responding differently than single-person households. Web visit frequency is higher among non-responders, highlighting an under-served web audience.

## 8. Define and Apply Rule-Based Segmentation

In [ ]:
p90_spend = clean['Total_Spend'].quantile(0.9)

conditions = [
    clean['Income'] > 75000,
    clean['Age'] < 30,
    clean['Response'] == 1,
    clean['NumWebVisitsMonth'] > 5,
    clean['Children'] > 0,
    clean['Total_Spend'] > p90_spend,
]
choices = [
    'High Income',
    'Young Customer',
    'Campaign Responder',
    'High Web Engagement',
    'Family Customer',
    'High Spender',
]
clean['Primary_Segment'] = np.select(conditions, choices, default='Regular')

# Multi-label membership columns
clean['High_Income'] = (clean['Income'] > 75000).astype(int)
clean['Young_Customer'] = (clean['Age'] < 30).astype(int)
clean['Campaign_Responder'] = (clean['Response'] == 1).astype(int)
clean['High_Web_Engagement'] = (clean['NumWebVisitsMonth'] > 5).astype(int)
clean['Family_Customer'] = (clean['Children'] > 0).astype(int)
clean['High_Spender'] = (clean['Total_Spend'] > p90_spend).astype(int)
clean['High_Value_Customer'] = (
    (clean['Income'] > 75000) & (clean['Total_Spend'] > p90_spend)
).astype(int)
clean['Under_Served_Customer'] = (
    (clean['Total_Spend'] <= clean['Total_Spend'].quantile(0.5))
    & (clean['NumWebVisitsMonth'] > 5)
    & (clean['Response'] == 0)
).astype(int)

clean['Primary_Segment'].value_counts().reset_index().rename(columns={'index': 'Segment', 'Primary_Segment': 'Count'})

**Takeaway:** The majority of customers fall into the "Family Customer" and "High Web Engagement" segments. "Regular" is the default catch-all. Multi-label flags allow each customer to belong to multiple segments simultaneously (e.g., a customer can be both High Income and High Spender), while `Primary_Segment` captures only the first matched rule for exclusive grouping.

## 9. Compute Segment-Level KPIs

In [ ]:
segment_kpis = clean.groupby('Primary_Segment').agg(
    total_customers=('ID', 'count'),
    avg_spend=('Total_Spend', 'mean'),
    response_rate=('Response', 'mean'),
    avg_income=('Income', 'mean'),
    avg_web_visits=('NumWebVisitsMonth', 'mean')
).reset_index()
segment_kpis['response_rate'] = segment_kpis['response_rate'] * 100
display(segment_kpis)

education_summary = clean.groupby('Education').agg(
    response_rate=('Response', 'mean'),
    avg_spend=('Total_Spend', 'mean'),
    count=('ID', 'count')
).reset_index()
education_summary['response_rate'] *= 100
display(education_summary)

country_summary = clean.groupby('Country').agg(
    response_rate=('Response', 'mean'),
    avg_spend=('Total_Spend', 'mean'),
    count=('ID', 'count')
).reset_index().sort_values('avg_spend', ascending=False)
display(country_summary.head(10))

age_band_summary = clean.groupby('Age_Band').agg(
    response_rate=('Response', 'mean'),
    avg_spend=('Total_Spend', 'mean'),
    count=('ID', 'count')
).reset_index()
age_band_summary['response_rate'] *= 100
display(age_band_summary)

income_band_summary = clean.groupby('Income_Band').agg(
    response_rate=('Response', 'mean'),
    avg_spend=('Total_Spend', 'mean'),
    count=('ID', 'count')
).reset_index()
income_band_summary['response_rate'] *= 100
display(income_band_summary)

**Takeaway:** High Income and Campaign Responder segments show the highest average spend and response rates, making them the most valuable targets. PhD and Master-educated customers tend to spend more and respond at higher rates. Spain and India appear among the top countries by spend. The 40–59 age bands contain the most engaged and highest-spending customers. Higher income bands consistently correlate with higher spend and better campaign response.

## 10. Prepare Cleaned Data for SQL and Dashboard Use

In [ ]:
import subprocess
import sys

# Export cleaned CSV (used by dashboard and build_sqlite_db.py)
output_csv = Path("..") / "cleaned_marketing_data.csv"
clean.to_csv(output_csv, index=False)
print('Exported cleaned data to', output_csv)

# Build SQLite database via build_sqlite_db.py which applies the full DDL schema
# (creates customers table + all analytical views: kpi_overview, segment_summary, etc.)
result = subprocess.run(
    [sys.executable, str(Path("..") / "build_sqlite_db.py")],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("build_sqlite_db.py error:", result.stderr)